# HARUM NOIR — ZERO COST STUDIO v1.1
Sem assinatura: FFmpeg CPU + Kokoro pt-BR + LTX-Video 2B opcional. Não depende de animação facial/deepfake.


In [ ]:
!apt-get -qq update
!apt-get -qq install -y ffmpeg espeak-ng fonts-dejavu-core
!pip -q install -U kokoro soundfile numpy pillow


In [ ]:
from google.colab import files
from pathlib import Path
uploaded=files.upload()
SOURCE=Path('/content')/next(iter(uploaded.keys()))


In [ ]:
import subprocess
DURATION=9
OUT=Path('/content/HARUM_NOIR_NOIRMOTION.mp4')
vf="scale=1200:2134:force_original_aspect_ratio=increase,crop=1200:2134,crop=1080:1920:x='(iw-ow)/2+20*sin(t*0.22)':y='(ih-oh)/2+12*sin(t*0.17)',eq=contrast=1.04:saturation=0.88:brightness=-0.015,vignette=PI/5,noise=alls=2:allf=t+u,fade=t=in:st=0:d=0.35,fade=t=out:st=8.5:d=0.5"
subprocess.run(['ffmpeg','-y','-loop','1','-i',str(SOURCE),'-t','9','-r','30','-vf',vf,'-an','-c:v','libx264','-crf','18','-pix_fmt','yuv420p','-movflags','+faststart',str(OUT)],check=True)


## Voz feminina pt-BR gratuita — Kokoro pf_dora


In [ ]:
import numpy as np, soundfile as sf
from kokoro import KPipeline
SCRIPT='Eu quase nunca sei quando um desenho terminou. Às vezes eu só paro antes de estragar.'
pipeline=KPipeline(lang_code='p')
chunks=[]
for _,_,audio in pipeline(SCRIPT,voice='pf_dora',speed=0.92): chunks.append(audio)
VOICE=Path('/content/harum_noir_voice.wav')
sf.write(VOICE,np.concatenate(chunks),24000)


In [ ]:
FINAL=Path('/content/HARUM_NOIR_2317_FINAL.mp4')
subprocess.run(['ffmpeg','-y','-i',str(OUT),'-i',str(VOICE),'-filter_complex','[1:a]volume=1.0,apad[a]','-map','0:v','-map','[a]','-c:v','copy','-c:a','aac','-b:a','192k','-shortest','-movflags','+faststart',str(FINAL)],check=True)
print(FINAL)


## LTX-Video 2B opcional
Use quando o Colab disponibilizar GPU. O fallback CPU acima continua funcionando.


In [ ]:
import torch
print('CUDA:',torch.cuda.is_available())
%cd /content
!rm -rf LTX-Video
!git clone -q --depth 1 https://github.com/Lightricks/LTX-Video.git
%cd /content/LTX-Video
!pip -q install -e .[inference]
